# KernelFuse Phase 4 — Nsight on Tier B

**Goal:** close the smem width gap. Occupancy alone predicts **2×** (4096→8192); Phase 3 measured **~2.7×** BW. Capture DRAM throughput + warp stall reasons (esp. barrier vs long scoreboard). Prefer a **T4 (sm_75)** so per-SM shared limits match the GTX 1650 story.

Runtime → change type → GPU. Then run all cells.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total,clocks.sm,clocks.mem,temperature.gpu --format=csv
!nvcc --version | tail -n 1
!which ncu || ls /usr/local/cuda*/bin/ncu 2>/dev/null; ls /opt/nvidia/nsight-compute/*/ncu 2>/dev/null | head

In [ ]:
# Clone or upload the repo, then cd into it.
import os
from pathlib import Path

ROOT = Path("/content/KernelFuse")
if not (ROOT / "kernels/rmsnorm/rmsnorm_fused_smem.cu").exists():
    # Replace with your fork URL if private.
    !git clone --depth 1 https://github.com/YOUR_USER/KernelFuse.git {ROOT}
os.chdir(ROOT)
print("cwd", Path.cwd())

In [ ]:
%%bash
chmod +x scripts/run_phase4_profile.sh
# T4 / 1650 = sm_75; A100 = sm_80; adjust if needed
export CUDA_ARCH=sm_75
./scripts/run_phase4_profile.sh

In [ ]:
from pathlib import Path
import pandas as pd

out = Path("reports/phase4")
print((out / "occupancy_probe.txt").read_text() if (out / "occupancy_probe.txt").exists() else "no occupancy yet")
for p in sorted(out.glob("*.csv")):
    print("\n===", p.name, "===")
    try:
        df = pd.read_csv(p, skiprows=0)
        # ncu CSV layout varies; show metric-ish columns if present
        cols = [c for c in df.columns if any(k in c.lower() for k in ("metric", "value", "kernel", "dram", "warp", "occup"))]
        display(df[cols].head(40) if cols else df.head(20))
    except Exception as e:
        print(p, e)
        print(p.read_text()[:2000])

## What to paste into `docs/phase_4_report.md`

1. Device name + clocks before/after (watch for throttle under ncu replay).
2. Occupancy probe: blocks/SM at 4096 vs 8192 (should be 4 vs 2 on sm_75).
3. Table: achieved occupancy %, `dram__bytes.sum.per_second`, barrier stall %, long_scoreboard stall % for `smem_4096`, `smem_8192`, `vec4_8192`, `fused_8192`.
4. Explicit sentence: predicted slowdown 2.0× from occupancy; measured BW ratio from Phase 3 was 2.7×; the extra factor is explained by ____ (fill from stalls/DRAM).
5. Download `reports/phase4/` and drop onto the laptop repo (gitignored under `reports/`).